# Lecture 5.3 — Handoff Input Filters: Controlling What the Next Agent Sees

In Lecture 5.2 you built handoffs with `handoff()`, used `on_handoff` callbacks, passed structured
data with `input_type`, and read the handoff trail from `result.new_items`. In every one of those
examples, the receiving agent saw the **entire** conversation history the moment a handoff fired.

In this notebook you will control that. You will use `input_filter` on `handoff()` to strip,
trim, or reshape exactly what the next agent sees, using both a built-in filter and filters you
write yourself.

**What you will build:**

1. A triage agent that hands off to a billing agent using the built-in `remove_all_tools` filter
2. A custom filter that composes `remove_all_tools` with your own trimming logic
3. A filter that uses `input_items` to control model input while preserving full session history
4. A global filter applied through `RunConfig.handoff_input_filter`
5. A look at `RECOMMENDED_PROMPT_PREFIX`, which helps the model understand handoffs correctly


## Cell 1: Install the OpenAI Agents SDK

This notebook uses the OpenAI Agents SDK's handoff system, specifically the `input_filter`
parameter and the built-in filters shipped in `agents.extensions.handoff_filters`. The cell below
installs the package.

If the package is already present in this Colab session, the install completes almost instantly
and confirms the existing version satisfies the pin.


In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.3/874.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.7 MB/s eta 0:00:00


## Cell 2: Configure Your OpenAI API Key

This notebook uses Google Colab Secrets to store your API key. Never paste an API key directly
into a notebook cell.

**Steps to add your key in Colab:**

1. Click the key icon (key emoji) in the left sidebar to open the **Secrets** panel.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.

Running the cell below reads that secret and writes it to an environment variable that the SDK
reads automatically.

**Local users:** if you are running this outside Colab, set the environment variable in your
terminal instead, for example `export OPENAI_API_KEY="sk-..."`, and skip the `userdata.get(...)`
call.


In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Set the Model

Every agent in this notebook references the `MODEL_NAME` variable below instead of a hardcoded
model string. Changing this one variable updates the model used across the entire notebook.

See the latest available models at the link in the comment before choosing a different one.


In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

| Import | Where it comes from | Why we need it |
|---|---|---|
| `Reasoning` | `openai.types.shared` | Sets `reasoning.effort` on `ModelSettings` |
| `Agent` | `agents` | Defines each agent in this notebook |
| `HandoffInputData` | `agents` | The frozen dataclass every `input_filter` receives and returns |
| `ModelSettings` | `agents` | Configures reasoning effort and verbosity per agent |
| `RunConfig` | `agents` | Lets us set a global `handoff_input_filter` for a run |
| `Runner` | `agents` | Executes agent runs with `await Runner.run(...)` |
| `function_tool` | `agents` | Turns a Python function into a tool the agent can call |
| `handoff` | `agents` | Builds a `Handoff` object, where `input_filter` is configured |
| `handoff_filters` | `agents.extensions` | A **separate submodule** holding built-in filters like `remove_all_tools` |
| `RECOMMENDED_PROMPT_PREFIX`, `prompt_with_handoff_instructions` | `agents.extensions.handoff_prompt` | Helper prefix and function that improve handoff routing accuracy |
| `MessageOutputItem` | `agents.items` | The run-item type representing a plain assistant message, used later to filter `new_items` down to messages only |
| `HandoffCallItem`, `HandoffOutputItem` | `agents.items` | The run-item types representing a handoff's tool call and its output, used later to isolate and remove handoff bookkeeping specifically, without touching regular tool items |

Two of these imports are worth calling out specifically because they live outside the top-level
`agents` package:

- `handoff_filters` comes from `agents.extensions`, not `agents` directly. Built-in filters are
  extensions, not core primitives.
- `RECOMMENDED_PROMPT_PREFIX` and `prompt_with_handoff_instructions` come from
  `agents.extensions.handoff_prompt`, a further submodule of extensions.

Both `HandoffInputData` and `MessageOutputItem` are importable from the top-level `agents` package
as well as their originating submodules (`agents.handoffs` and `agents.items` respectively). This
notebook imports `HandoffInputData` from `agents` for convenience and `MessageOutputItem` from
`agents.items` to keep the item-type imports together.


In [4]:
from openai.types.shared import Reasoning
from agents import (
    Agent,
    HandoffInputData,
    ModelSettings,
    RunConfig,
    Runner,
    function_tool,
    handoff,
)
from agents.extensions import handoff_filters
from agents.extensions.handoff_prompt import (
    RECOMMENDED_PROMPT_PREFIX,
    prompt_with_handoff_instructions,
)
from agents.items import HandoffCallItem, HandoffOutputItem, MessageOutputItem

## Cell 5: Why Input Filters Exist

By default, when a handoff occurs the new agent sees the **entire** conversation history, as
though it had been part of the conversation from the start. Every user message, every tool call,
every tool output the previous agent generated becomes visible to the agent that takes over.

That is often more than the specialist needs. A billing agent that receives a handoff does not
need to see the order-lookup tool call and its output that the triage agent made two turns
earlier. It just needs to know what the customer is asking for now.

`input_filter` is how you change what the next agent sees. It is a function that:

- Receives the current `HandoffInputData`
- Must return a **new** `HandoffInputData`

### The frozen dataclass

`HandoffInputData` is declared as `@dataclass(frozen=True)`. That means it is **immutable**. You
cannot do `handoff_input_data.new_items = ()` and expect it to work. Every filter must produce a
modified copy using `.clone(**kwargs)`, which internally uses `dataclasses.replace()`. This keeps
filters composable and predictable: a filter always takes one `HandoffInputData` in and returns
one out, with no risk of a filter accidentally mutating state another filter still depends on.

### A critical streaming note

Straight from the SDK's docstring on `Handoff.input_filter`:

> "IMPORTANT: in streaming mode, we will not stream anything as a result of this function. The
> items generated before will already have been streamed."

In other words, if you are streaming a run and a handoff with an `input_filter` fires mid-stream,
the filtering itself produces no new stream events. The items that came before the handoff were
already streamed to the caller before the filter ran.

### A related restriction worth knowing

The same docstring also notes that **server-managed conversations** (runs using `conversation_id`,
`previous_response_id`, or `auto_previous_response_id`) do not support handoff input filters at
all. Input filters assume the SDK has the full local transcript to filter. When the conversation
state instead lives on OpenAI's servers, there is no local transcript for the filter to operate
on. This notebook uses locally-managed runs throughout, so every filter below applies normally.

### Priority rule

A handoff can get its filter from two places: a per-handoff `input_filter` passed to `handoff()`,
or a run-wide `RunConfig.handoff_input_filter`. When both are set for a given handoff, the
**per-handoff `input_filter` wins**. You will see both in this notebook, and the priority rule
demonstrated directly in the RunConfig section.

### Scope reminder

Also worth keeping in mind, from the docs:

> "Handoffs remain within a single run. Input guardrails still apply only to the first agent in
> the chain, and output guardrails apply only to the agent that produces the final output."

Input filters reshape what the next agent sees, but they do not change which agent guardrails run
against. That stays fixed regardless of how many handoffs occur or how their inputs are filtered.


## Cell 6: `HandoffInputData` Field Reference

Every filter function receives one of these and must return one. Here is what each field holds:

| Field | Type | Description |
|---|---|---|
| `input_history` | `str \| tuple[TResponseInputItem, ...]` | The input history before `Runner.run(...)` started. |
| `pre_handoff_items` | `tuple[RunItem, ...]` | Items generated **before** the agent turn where the handoff was invoked. |
| `new_items` | `tuple[RunItem, ...]` | Items generated **during** the current turn, including the handoff call and handoff output items. |
| `run_context` | `RunContextWrapper[Any] \| None` | The active run context at the time the handoff was invoked. Optional, for backwards compatibility. |
| `input_items` | `tuple[RunItem, ...] \| None` | Optional. When set, it is used **instead of** `new_items` for building the next agent's input. `new_items` stays untouched for session history. |

Two fields are easy to confuse at first glance: `new_items` and `input_items`.

- `new_items` is always the full record of what happened this turn. It is what gets saved to
  session history, and filters that only touch `new_items` directly change what the model sees
  **and** what gets remembered.
- `input_items`, when set, lets you separate those two concerns. The model sees `input_items`.
  Session history still gets the untouched `new_items`. You will see this pattern directly in the
  `input_items` section below.

The only way to produce a modified `HandoffInputData` is `clone(**kwargs)` — the dataclass is
frozen, so direct field assignment raises an error.


## Cell 7: Seeing the Fields Live

The table above is the reference. Here is what those fields actually look like on a real run.
The filter below, `inspect_handoff_data`, changes nothing. It just prints every field on
`HandoffInputData` and returns the object unmodified, to show that a filter is not required to
alter what it receives.

The scenario: a triage agent checks membership status with a tool, then hands off to a specialist
in a later turn. That's deliberate. It's the only way to get something into `pre_handoff_items`,
since anything from an earlier turn than the handoff itself lands there rather than in
`new_items`. Watch how `input_history`, `pre_handoff_items`, and `new_items` differ in shape, and
notice that `input_items` prints as `None` here. Nothing has set it yet. That field only gets used
starting in Cell 10.


In [5]:
@function_tool
def check_membership(account_id: str) -> str:
    """Checks a customer's membership status.

    Args:
        account_id: The account ID to check.
    """
    return f"Account {account_id}: active membership since 2022."


def inspect_handoff_data(
    handoff_input_data: HandoffInputData,
) -> HandoffInputData:
    """
    Read-only filter: prints every field on HandoffInputData
    without changing anything. Returned unmodified, since a
    filter is not required to alter the data it receives.
    """
    print("input_history:")
    history = handoff_input_data.input_history
    if isinstance(history, str):
        print(f"  (str) {history!r}")
    else:
        for item in history:
            print(f"  {item}")

    print("\npre_handoff_items:")
    if handoff_input_data.pre_handoff_items:
        for item in handoff_input_data.pre_handoff_items:
            raw = getattr(item, "raw_item", None)
            print(f"  {type(item).__name__} -> {raw}")
    else:
        print("  (empty)")

    print("\nnew_items:")
    for item in handoff_input_data.new_items:
        raw = getattr(item, "raw_item", None)
        print(f"  {type(item).__name__} -> {raw}")

    print("\nrun_context:")
    run_context = handoff_input_data.run_context
    print(f"  type: {type(run_context).__name__}")
    if run_context is not None and hasattr(run_context, "context"):
        print(f"  .context: {run_context.context!r}")

    print("\ninput_items:")
    print(f"  {handoff_input_data.input_items}")

    return handoff_input_data


field_demo_specialist = Agent(
    name="Specialist Agent",
    instructions=(
        "You are a specialist. "
        "Answer the user's current question."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

field_demo_triage = Agent(
    name="Field Demo Triage",
    instructions=(
        "You are a triage agent. "
        "Check membership using check_membership when the user "
        "gives an account ID. "
        "Then hand off to the Specialist Agent for further help."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[check_membership],
    handoffs=[
        handoff(
            field_demo_specialist,
            input_filter=inspect_handoff_data,
        ),
    ],
)

result = await Runner.run(
    field_demo_triage,
    "My account ID is ACC-900. I need help with my subscription.",
)

print("\nFinal output:", result.final_output)
print("Last agent:", result.last_agent.name)

input_history:
  (str) 'My account ID is ACC-900. I need help with my subscription.'

pre_handoff_items:
  ToolCallItem -> ResponseFunctionToolCall(arguments='{"account_id":"ACC-900"}', call_id='call_VJ5VMr2h3TE0toT7zHIjZ05t', name='check_membership', type='function_call', id='fc_05197dd753dc8f4f006a7be35bf120819ab88a51e91ecc951b', caller=None, namespace=None, status='completed')
  ToolCallOutputItem -> {'call_id': 'call_VJ5VMr2h3TE0toT7zHIjZ05t', 'output': 'Account ACC-900: active membership since 2022.', 'type': 'function_call_output'}

new_items:
  HandoffCallItem -> ResponseFunctionToolCall(arguments='{}', call_id='call_4Be9gIfVVeIpZQG7zEFxq3Mn', name='transfer_to_specialist_agent', type='function_call', id='fc_05197dd753dc8f4f006a7be35d6cf0819ab7a1c63aa66d0e61', caller=None, namespace=None, status='completed')
  HandoffOutputItem -> {'call_id': 'call_4Be9gIfVVeIpZQG7zEFxq3Mn', 'output': '{"assistant": "Specialist Agent"}', 'type': 'function_call_output'}

run_context:
  type: RunC

## Cell 8: The Built-In `remove_all_tools` Filter

`agents.extensions.handoff_filters.remove_all_tools` is the SDK's built-in filter for the most
common case: strip every tool-related item out of the history before the next agent sees it. It
removes tool calls and their outputs, hosted tool calls (web search, file search, code
interpreter), reasoning items, and MCP-related items, from `input_history`, `pre_handoff_items`,
`new_items`, and `input_items` alike.

The scenario below defines a `lookup_order` tool on a triage agent. The triage agent uses that
tool, then hands off to a billing agent using `input_filter=handoff_filters.remove_all_tools`.
The billing agent will see that the customer looked something up and now needs billing help, but
it will not see the `lookup_order` call or its raw output.

**What this filter does and does not touch.** `remove_all_tools` only removes tool-related items.
It does not touch plain user or assistant messages. In the scenario below, the user's original
message, "Look up order ORD-001. Now I need to update my payment method," is a plain message item,
not a tool item, so it survives the filter untouched. The `lookup_order` tool call and its output,
"Order ORD-001: 2x Widget Pro, shipped, ETA 3 days," are tool items, so the filter strips both of
them out before the handoff.

That distinction is exactly what you should expect to see in the output below. The billing agent
will know the conversation is about order ORD-001 and a payment method update, because it can
still read the user's original message. It will have no idea the order was "2x Widget Pro,
shipped, ETA 3 days," because that tool output never reaches it. If the billing agent's response
asks you to verify the order or the account rather than confirming order details directly, that is
the filter working as intended. It has conversational context, not the tool's data.


In [6]:
@function_tool
def lookup_order(order_id: str) -> str:
    """Looks up an order by ID.

    Args:
        order_id: The order ID to look up.
    """
    return (
        f"Order {order_id}: 2x Widget Pro, shipped, "
        f"ETA 3 days."
    )


billing_agent = Agent(
    name="Billing Agent",
    instructions=(
        "You are a billing specialist. "
        "Help users with payment and invoice questions."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

triage_with_filter = Agent(
    name="Triage Agent (with filter)",
    instructions=(
        "You are a triage agent. "
        "Help users with order lookups using the "
        "lookup_order tool. "
        "If the user needs billing help, hand off to "
        "the Billing Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[lookup_order],
    handoffs=[
        handoff(
            billing_agent,
            input_filter=handoff_filters.remove_all_tools,
        ),
    ],
)

result = await Runner.run(
    triage_with_filter,
    "Look up order ORD-001. Now I need to update my "
    "payment method.",
)
print("Final output:", result.final_output)
print("Last agent:", result.last_agent.name)

Final output: Order ORD-001: 2x Widget Pro, shipped, ETA 3 days.

For updating your payment method, I’m transferring you to the Billing Agent.
Last agent: Triage Agent (with filter)


## Cell 9: Writing Custom `input_filter`s, and What They Actually Cost You

Built-in filters cover the common case, but nothing stops you from writing your own. A custom
filter is a plain Python function: it receives a `HandoffInputData` and returns a
`HandoffInputData`. Both sync and async filter functions are supported by the SDK.

This cell builds two custom filters and runs the same scenario three times so you can see exactly
what each choice costs:

- **`clean_history_filter`** composes `remove_all_tools` with extra trimming logic on top: it
  strips tool items exactly like the previous cell, then trims `input_history` down to the last
  two items. Like `remove_all_tools`, it strips every tool item, including the account lookup
  tool's result.
- **`handoff_only_filter`** does the opposite selection. It strips only the handoff bookkeeping
  items, `HandoffCallItem` and `HandoffOutputItem`, using an `isinstance()` check, and leaves every
  regular tool call and its result completely untouched.

Both filters print what they receive, what they remove, and what they hand back, so you can watch
`HandoffInputData`'s collections, `input_history`, `pre_handoff_items`, and `new_items`, change
shape at each step. Notice that both filters call `.clone()` to produce their result. Every step
that changes the data produces a new `HandoffInputData`. Nothing is mutated in place, since the
dataclass is frozen.

The scenario behind all three runs: a triage agent looks up an account's tier and spending limit
with `lookup_account_tier`, then hands off to a specialist because the user wants to raise that
limit. The specialist needs the current limit to act on the request. Run 1 hands off with no
filter at all, the SDK default, so the specialist sees everything, tool result included. Run 2
uses `clean_history_filter`, so the specialist loses the tool result along with the handoff
bookkeeping. Run 3 uses `handoff_only_filter`, so the specialist keeps the tool result and only
loses the handoff bookkeeping.

Watch the final side-by-side print at the end of the cell. Expect Run 1 and Run 3 to land close
together, since both preserve the account's actual spending limit for the specialist, while Run 2
should be the odd one out. Without the tool result, the specialist has no figure to act on and
will likely ask for it instead of confirming a new limit directly.


In [7]:
def print_history(label, history):
    print(f"  {label}:")
    if isinstance(history, str):
        print(f"    {history!r}")
    else:
        for item in history:
            print(f"    {item}")


def clean_history_filter(
    handoff_input_data: HandoffInputData,
) -> HandoffInputData:
    """
    Custom filter: remove all tools AND trim input history
    to last 2 items. Demonstrates composing remove_all_tools
    with custom logic.
    """
    print("HandoffInputData received:")
    print_history("input_history", handoff_input_data.input_history)
    print("  pre_handoff_items:")
    for item in handoff_input_data.pre_handoff_items:
        print(f"    {type(item).__name__}")
    print("  new_items:")
    for item in handoff_input_data.new_items:
        print(f"    {type(item).__name__}")

    data = handoff_filters.remove_all_tools(handoff_input_data)

    removed_pre_handoff = [
        type(item).__name__
        for item in handoff_input_data.pre_handoff_items
        if item not in data.pre_handoff_items
    ]
    removed_new_items = [
        type(item).__name__
        for item in handoff_input_data.new_items
        if item not in data.new_items
    ]
    print(f"\nRemoved by remove_all_tools:")
    print(f"  from pre_handoff_items: {removed_pre_handoff if removed_pre_handoff else 'nothing'}")
    print(f"  from new_items: {removed_new_items if removed_new_items else 'nothing'}")

    print("\nLeft after remove_all_tools:")
    print_history("input_history", data.input_history)
    print("  pre_handoff_items:")
    for item in data.pre_handoff_items:
        print(f"    {type(item).__name__}")
    print("  new_items:")
    for item in data.new_items:
        print(f"    {type(item).__name__}")

    history = data.input_history
    if isinstance(history, tuple) and len(history) > 2:
        removed_history = history[:-2]
        history = history[-2:]
        print(f"\nTrimmed from input_history (kept only last 2): {removed_history}")
    else:
        print("\nTrimmed from input_history: nothing (2 or fewer items, or a plain string)")

    result = data.clone(input_history=history)

    print("\nFinal items passed to the next agent:")
    print_history("input_history", result.input_history)
    print("  pre_handoff_items:")
    for item in result.pre_handoff_items:
        print(f"    {type(item).__name__}")
    print("  new_items:")
    for item in result.new_items:
        print(f"    {type(item).__name__}")

    return result


def handoff_only_filter(
    handoff_input_data: HandoffInputData,
) -> HandoffInputData:
    """
    Custom filter: remove ONLY the handoff bookkeeping items
    (HandoffCallItem, HandoffOutputItem). Every regular tool
    call and its output stays intact, unlike remove_all_tools.
    """
    print("HandoffInputData received:")
    print("  pre_handoff_items:")
    for item in handoff_input_data.pre_handoff_items:
        print(f"    {type(item).__name__}")
    print("  new_items:")
    for item in handoff_input_data.new_items:
        print(f"    {type(item).__name__}")

    def drop_handoff_items(items):
        return tuple(
            item
            for item in items
            if not isinstance(item, (HandoffCallItem, HandoffOutputItem))
        )

    filtered_pre_handoff = drop_handoff_items(handoff_input_data.pre_handoff_items)
    filtered_new_items = drop_handoff_items(handoff_input_data.new_items)

    removed_new_items = [
        type(item).__name__
        for item in handoff_input_data.new_items
        if item not in filtered_new_items
    ]
    print(f"\nRemoved by handoff_only_filter (from new_items): {removed_new_items if removed_new_items else 'nothing'}")

    result = handoff_input_data.clone(
        pre_handoff_items=filtered_pre_handoff,
        new_items=filtered_new_items,
    )

    print("\nFinal items passed to the next agent:")
    print("  pre_handoff_items:")
    for item in result.pre_handoff_items:
        print(f"    {type(item).__name__}")
    print("  new_items:")
    for item in result.new_items:
        print(f"    {type(item).__name__}")

    return result


@function_tool
def lookup_account_tier(account_id: str) -> str:
    """Looks up an account's membership tier and spending limit.

    Args:
        account_id: The account ID to look up.
    """
    return (
        f"Account {account_id}: Gold tier, "
        f"$500 monthly spending limit, priority support enabled."
    )


specialist_agent = Agent(
    name="Specialist Agent",
    instructions=(
        "You are a specialist. "
        "Answer the user's current question."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

# Run 1: no input_filter at all. Specialist sees everything.
unfiltered_triage = Agent(
    name="Unfiltered Triage",
    instructions=(
        "You are a triage agent. "
        "Look up the account tier using lookup_account_tier "
        "when the user gives an account ID. "
        "Then hand off to the Specialist Agent for further help."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[lookup_account_tier],
    handoffs=[specialist_agent],  # no input_filter here
)

# Run 2: remove_all_tools (via clean_history_filter). Specialist
# loses the tool call AND the handoff bookkeeping.
filtered_triage = Agent(
    name="Filtered Triage",
    instructions=(
        "You are a triage agent. "
        "Look up the account tier using lookup_account_tier "
        "when the user gives an account ID. "
        "Then hand off to the Specialist Agent for further help."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[lookup_account_tier],
    handoffs=[
        handoff(
            specialist_agent,
            input_filter=clean_history_filter,
        ),
    ],
)

# Run 3: handoff_only_filter. Specialist keeps the tool call and
# its result, loses only the handoff bookkeeping.
handoff_only_triage = Agent(
    name="Handoff-Only-Filtered Triage",
    instructions=(
        "You are a triage agent. "
        "Look up the account tier using lookup_account_tier "
        "when the user gives an account ID. "
        "Then hand off to the Specialist Agent for further help."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[lookup_account_tier],
    handoffs=[
        handoff(
            specialist_agent,
            input_filter=handoff_only_filter,
        ),
    ],
)

user_message = "My account ID is ACC-500. I need to raise my spending limit."

print("=" * 60)
print("RUN 1: no input_filter (specialist sees everything)")
print("=" * 60)
unfiltered_result = await Runner.run(unfiltered_triage, user_message)
print("Final output:", unfiltered_result.final_output)
print("Last agent:", unfiltered_result.last_agent.name)

print("\n" + "=" * 60)
print("RUN 2: remove_all_tools (specialist loses the tool call too)")
print("=" * 60)
filtered_result = await Runner.run(filtered_triage, user_message)
print("Final output:", filtered_result.final_output)
print("Last agent:", filtered_result.last_agent.name)

print("\n" + "=" * 60)
print("RUN 3: handoff_only_filter (specialist keeps the tool call)")
print("=" * 60)
handoff_only_result = await Runner.run(handoff_only_triage, user_message)
print("Final output:", handoff_only_result.final_output)
print("Last agent:", handoff_only_result.last_agent.name)

print("\n" + "=" * 60)
print("SIDE BY SIDE")
print("=" * 60)
print("No filter:          ", unfiltered_result.final_output)
print("\nremove_all_tools:    ", filtered_result.final_output)
print("\nhandoff_only_filter: ", handoff_only_result.final_output)

RUN 1: no input_filter (specialist sees everything)
Final output: Your account **ACC-500** is currently **Gold tier** with a **$500 monthly spending limit**.

I’ve escalated this to a specialist agent to help with raising your limit.
Last agent: Specialist Agent

RUN 2: remove_all_tools (specialist loses the tool call too)
HandoffInputData received:
  input_history:
    'My account ID is ACC-500. I need to raise my spending limit.'
  pre_handoff_items:
    ToolCallItem
    ToolCallOutputItem
  new_items:
    HandoffCallItem
    HandoffOutputItem

Removed by remove_all_tools:
  from pre_handoff_items: ['ToolCallItem', 'ToolCallOutputItem']
  from new_items: ['HandoffCallItem', 'HandoffOutputItem']

Left after remove_all_tools:
  input_history:
    'My account ID is ACC-500. I need to raise my spending limit.'
  pre_handoff_items:
  new_items:

Trimmed from input_history: nothing (2 or fewer items, or a plain string)

Final items passed to the next agent:
  input_history:
    'My account

## Cell 10: `input_items`: Filtering Model Input While Preserving History

Cell 8 and Cell 9 both answered the same question: *which items get removed?* `remove_all_tools`
and `handoff_only_filter` return smaller versions of `pre_handoff_items` and `new_items`. Whatever
gets dropped is gone for good, from the model's view and from the session record alike.

This cell shows a different capability: showing the model one view while the record stays intact.

`HandoffInputData` has a field called `input_items`, separate from `new_items`. Here is the rule,
straight from how the SDK builds the next agent's input:

| `input_items` | What builds the next agent's input | What goes to session history |
|---|---|---|
| `None` (default) | `new_items` | `new_items` |
| Set to something | `input_items` | `new_items` (untouched) |

One detail matters here, and it is easy to get wrong: `input_items` only ever substitutes for
`new_items`. It has **no effect on `pre_handoff_items`**, which always flows through to the next
agent exactly as filtered elsewhere. If you want `input_items` to visibly change what the model
sees, the content you are hiding has to originate in `new_items`, meaning it has to happen in the
same turn as the handoff itself, not an earlier one.

That is why the triage agent below is instructed to say one line to the user before handing off.
That message and the handoff call both land in `new_items` together. The filter,
`filter_to_messages_only`, keeps only the `MessageOutputItem` and drops the handoff bookkeeping
(`HandoffCallItem`, `HandoffOutputItem`) from what the specialist's model call actually reads, by
routing just the message through `input_items`. `new_items` itself is returned completely
unchanged. Check `result.new_items` on the top-level `RunResult` after the run finishes. It still
holds everything, message and handoff items alike, because `input_filter` only ever controls what
one agent's model call sees, never what the SDK records for the run as a whole.


In [8]:
def filter_to_messages_only(
    handoff_input_data: HandoffInputData,
) -> HandoffInputData:
    """
    Use input_items to pass only message items to the next
    agent, while new_items preserves everything for session
    history.
    """
    print("HandoffInputData received - new_items:")
    for item in handoff_input_data.new_items:
        print(f"  {type(item).__name__}")

    message_items = tuple(
        item for item in handoff_input_data.new_items
        if isinstance(item, MessageOutputItem)
    )
    print(
        "\ninput_items being set to (messages only):",
        [type(i).__name__ for i in message_items] or "nothing, empty tuple",
    )

    result = handoff_input_data.clone(input_items=message_items)

    print("\nnew_items on the returned HandoffInputData (untouched, still has everything):")
    for item in result.new_items:
        print(f"  {type(item).__name__}")

    return result


specialist_agent_input_items = Agent(
    name="Specialist Agent",
    instructions=(
        "You are a specialist. "
        "Answer the user's current question."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

input_items_triage = Agent(
    name="Input Items Triage",
    instructions=(
        "You are a triage agent. "
        "Before handing off, always tell the user in one short "
        "sentence that you're connecting them to a specialist. "
        "Then hand off to the Specialist Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[
        handoff(
            specialist_agent_input_items,
            input_filter=filter_to_messages_only,
        ),
    ],
)

result = await Runner.run(
    input_items_triage,
    "I need specialist help with my account.",
)

print("\nFinal output:", result.final_output)
print("Last agent:", result.last_agent.name)

print("\nresult.new_items on the overall RunResult (record still has everything):")
for item in result.new_items:
    print(f"  {type(item).__name__}")

HandoffInputData received - new_items:
  MessageOutputItem
  HandoffCallItem
  HandoffOutputItem

input_items being set to (messages only): ['MessageOutputItem']

new_items on the returned HandoffInputData (untouched, still has everything):
  MessageOutputItem
  HandoffCallItem
  HandoffOutputItem

Final output: I can help with account issues.

Please tell me:
- what account type this is
- what’s happening
- any error message you see
- when it started

If you want, I can also help you with:
- login/access problems
- password reset
- billing/subscription issues
- account recovery
- security/suspicious activity
Last agent: Specialist Agent

result.new_items on the overall RunResult (record still has everything):
  MessageOutputItem
  HandoffCallItem
  HandoffOutputItem
  MessageOutputItem


## Cell 11: `RunConfig.handoff_input_filter`: A Global Policy

Every filter so far was set on a specific `handoff()` call, so it only applies to that one
handoff. `RunConfig.handoff_input_filter` sets a **global** filter for the entire run instead. It
applies to every handoff in that run that does **not** already have its own per-handoff filter.

Recall the priority rule from Cell 5: `input_filter = handoff.input_filter or
run_config.handoff_input_filter`. Because it's an `or`, exactly one of the two functions ever
runs for a given handoff, never both. This cell is built to prove that directly, not just assert
it.

**The setup.** Two triage agents share the same tool, `check_order_status`, and the same billing
agent. `triage_with_own_filter`'s handoff has its own `input_filter`,
`handoff_filters.remove_all_tools`, which strips the tool call before the specialist sees it.
`triage_without_own_filter`'s handoff has no `input_filter` at all. Both runs pass the exact same
`RunConfig`, carrying `handoff_input_filter=run_config_filter`, a filter defined specifically for
this cell that does **not** strip anything. It returns `handoff_input_data` completely unchanged,
and prints a marker line so you can see exactly when it executes.

That asymmetry is the point. If `remove_all_tools` and `run_config_filter` both stripped tool
calls, the two runs would look identical and you'd have no way to tell which filter actually ran
just by reading the output. Making them behave differently turns the priority rule into something
visible: whichever filter actually executes leaves its own fingerprint on the specialist's
response.

**What to expect.** In Run 1, `triage_with_own_filter`'s own `input_filter` wins, so the tool
result gets stripped and the specialist has no idea what the order status was. `run_config_filter`
never runs at all here, so its marker line should be **missing** from the output. In Run 2,
`triage_without_own_filter`'s handoff has nothing of its own, so `run_config_filter` steps in,
its marker line **does** print, and because it passes everything through untouched, the specialist
should be able to state the order's actual status directly.

Run the cell and check both signals together: the marker line's presence or absence, and whether
the specialist's answer references real order details or not. They should agree with each other
every time, and that agreement is the proof that the per-handoff filter takes priority over the
RunConfig-level one exactly as the rule says.

In [9]:
@function_tool
def check_order_status(order_id: str) -> str:
    """Checks the status of an order by ID.

    Args:
        order_id: The order ID to check.
    """
    return f"Order {order_id}: 2x Widget Pro, shipped, ETA 3 days."


def run_config_filter(
    handoff_input_data: HandoffInputData,
) -> HandoffInputData:
    """
    RunConfig-level filter. Deliberately does NOT strip anything,
    so its effect on the output is visibly different from the
    per-handoff filter below. Prints a marker too, so you have
    both a content-level and a print-level way to tell it ran.
    """
    print(">>> run_config_filter (the RunConfig-level filter) is running <<<")
    return handoff_input_data  # unchanged: no stripping at all


rc_demo_billing_agent = Agent(
    name="Billing Agent",
    instructions=(
        "You are a billing specialist. "
        "Help users with payment and invoice questions."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

# Run 1: this handoff HAS its own per-handoff filter, which strips
# tool items. Expect the specialist to lose the order details.
triage_with_own_filter = Agent(
    name="Triage With Own Filter",
    instructions=(
        "You are a triage agent. "
        "Help users with order lookups using the "
        "check_order_status tool. "
        "If the user needs billing help, hand off to "
        "the Billing Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[check_order_status],
    handoffs=[
        handoff(
            rc_demo_billing_agent,
            input_filter=handoff_filters.remove_all_tools,
        ),
    ],
)

# Run 2: this handoff has NO per-handoff filter, so RunConfig's
# pass-through filter takes over. Expect the specialist to KEEP
# the order details, since run_config_filter strips nothing.
triage_without_own_filter = Agent(
    name="Triage Without Own Filter",
    instructions=(
        "You are a triage agent. "
        "Help users with order lookups using the "
        "check_order_status tool. "
        "If the user needs billing help, hand off to "
        "the Billing Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[check_order_status],
    handoffs=[rc_demo_billing_agent],  # no input_filter set on this handoff
)

print("=" * 60)
print("RUN 1: triage_with_own_filter (per-handoff filter wins, strips tools)")
print("=" * 60)
result_1 = await Runner.run(
    triage_with_own_filter,
    "Check order ORD-003. Now I need billing help.",
    run_config=RunConfig(
        workflow_name="Global filter demo",
        handoff_input_filter=run_config_filter,
    ),
)
print("Final output:", result_1.final_output)
print("Last agent:", result_1.last_agent.name)

print("\n" + "=" * 60)
print("RUN 2: triage_without_own_filter (RunConfig filter wins, keeps tools)")
print("=" * 60)
result_2 = await Runner.run(
    triage_without_own_filter,
    "Check order ORD-004. Now I need billing help.",
    run_config=RunConfig(
        workflow_name="Global filter demo",
        handoff_input_filter=run_config_filter,
    ),
)
print("Final output:", result_2.final_output)
print("Last agent:", result_2.last_agent.name)

print("\n" + "=" * 60)
print("SIDE BY SIDE")
print("=" * 60)
print("Per-handoff filter won:  ", result_1.final_output)
print("\nRunConfig filter won:    ", result_2.final_output)

RUN 1: triage_with_own_filter (per-handoff filter wins, strips tools)
Final output: I can help with billing, but I can’t directly look up **ORD-003** from here.

Please send one of these:
- the **invoice/billing issue** you’re seeing,
- a **copy of the invoice** or order details,
- or the **specific question** (charge amount, tax, refund, duplicate charge, payment failed, etc.).

If you want, I can also help you draft a billing support request for **ORD-003**.
Last agent: Billing Agent

RUN 2: triage_without_own_filter (RunConfig filter wins, keeps tools)
>>> run_config_filter (the RunConfig-level filter) is running <<<
Final output: I’ve checked it: **ORD-004** is shipped and due in **3 days**.

I’ve also transferred you to the **Billing Agent** for help with billing.
Last agent: Billing Agent

SIDE BY SIDE
Per-handoff filter won:   I can help with billing, but I can’t directly look up **ORD-003** from here.

Please send one of these:
- the **invoice/billing issue** you’re seeing,
- a

## Cell 12: `RECOMMENDED_PROMPT_PREFIX`: Helping the Model Understand Handoffs

Input filters control what history the model sees. `RECOMMENDED_PROMPT_PREFIX` addresses a
different problem: making sure the model itself understands what a handoff is and how to use one
correctly.

`RECOMMENDED_PROMPT_PREFIX` is a standard block of instruction text the SDK docs recommend
including in the instructions of any agent that has handoffs. `prompt_with_handoff_instructions(
instructions)` is a small helper that prepends this prefix to your own instructions automatically,
so you do not have to concatenate it by hand every time.

Neither is mandatory. An agent will still route handoffs without them. But the docs recommend
including one or the other because it measurably improves routing accuracy, especially once a
system has more than one or two possible handoff targets.


In [10]:
print("RECOMMENDED_PROMPT_PREFIX:")
print(RECOMMENDED_PROMPT_PREFIX)

agent_instructions = "You are a helpful triage agent."
full_instructions = prompt_with_handoff_instructions(
    agent_instructions
)
print("\nWith handoff instructions prepended:")
print(full_instructions)

RECOMMENDED_PROMPT_PREFIX:
# System context
You are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy. Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function, generally named `transfer_to_<agent_name>`. Transfers between agents are handled seamlessly in the background; do not mention or draw attention to these transfers in your conversation with the user.


With handoff instructions prepended:
# System context
You are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy. Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function, generally

## Cell 13: `input_filter` Reference

A quick reference for choosing the right filtering approach:

| What to filter | Tool | Notes |
|---|---|---|
| All tool calls/outputs | `handoff_filters.remove_all_tools` | Removes function calls, web/file search, code interpreter, reasoning, MCP items, handoff items |
| Custom subset | Custom function with `.clone()` | Inspect `new_items` / `pre_handoff_items`, filter by `isinstance()` |
| Trim conversation length | Custom function slicing `input_history` | Reduces token cost for the receiving agent |
| Replace history with a summary | Custom function modifying `input_history` | Pass a summarised string instead of a tuple |
| Control model input vs. session record | Set `input_items` separately from `new_items` | Model sees `input_items`; session stores `new_items` |

**Priority rule:** per-handoff `input_filter` takes precedence over `RunConfig.handoff_input_filter`
for that handoff. Both sync and async filter functions are supported.

**What this notebook did not cover:** `nest_handoff_history` / `RunConfig.nest_handoff_history`
(an opt-in beta feature), `HandoffHistoryMapper` / `default_handoff_history_mapper`,
`set_conversation_history_wrappers`, and guardrails on handoffs. Guardrails are covered in
Lectures 5.8 through 5.10.
